In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops

In [ ]:
pd.set_option('display.max_rows', 80)

In [ ]:
# Load NSL-KDD
df = pd.read_csv('CICIDS_combined.csv', header=None)


FileNotFoundError: [Errno 2] No such file or directory: 'CICIDS_combined.csv'

In [ ]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,69,70,71,72,73,74,75,76,77,78
0,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
1,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
3,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
4,46236,34,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN


In [ ]:
df.columns

Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
       36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
       54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
       72, 73, 74, 75, 76, 77, 78],
      dtype='int64')

In [ ]:
df.head()

In [ ]:
temp_columns = df.iloc[0,:]

In [ ]:
temp_columns[0]

' Destination Port'

In [ ]:
columns  = [column.strip() for column in temp_columns]

In [ ]:
columns

['Destination Port',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Total Length of Fwd Packets',
 'Total Length of Bwd Packets',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min',
 'Fwd IAT Total',
 'Fwd IAT Mean',
 'Fwd IAT Std',
 'Fwd IAT Max',
 'Fwd IAT Min',
 'Bwd IAT Total',
 'Bwd IAT Mean',
 'Bwd IAT Std',
 'Bwd IAT Max',
 'Bwd IAT Min',
 'Fwd PSH Flags',
 'Bwd PSH Flags',
 'Fwd URG Flags',
 'Bwd URG Flags',
 'Fwd Header Length',
 'Bwd Header Length',
 'Fwd Packets/s',
 'Bwd Packets/s',
 'Min Packet Length',
 'Max Packet Length',
 'Packet Length Mean',
 'Packet Length Std',
 'Packet Length Variance',
 'FIN Flag Count',
 'SYN Flag Count',
 'RST Flag Count',
 'PSH Flag Count',
 'ACK Flag Count',
 'UR

In [ ]:
len(columns)

79

In [ ]:
df.columns = columns

In [ ]:
df = df.drop(index = 0)

In [ ]:
# Load the NSL-KDD dataset (change the path if needed)

# Assign meaningful column names (use real NSL-KDD names if you have them)
df.columns = columns

# Encode categorical features
categorical_cols = ['protocol_type', 'service', 'flag']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Convert labels to binary: 0 = normal, 1 = anomaly
df['label'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)

# Select a subset of features (you can expand this later)
X = df.drop('label',axis=1)
y = df['label']


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN


In [ ]:
df.iloc[[0]]

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN


In [ ]:
df.apply(lambda col: col.astype(str).str.strip())

In [ ]:
df.head()

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Columns to exclude from conversion (already int)
exclude_cols = ['protocol_type', 'service', 'flag']

# Convert all other columns in X to float
for col in X.columns:
    if col not in exclude_cols:
        X[col] = X[col].astype(float)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split dataset into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize the scaler
scaler = StandardScaler()

# Fit scaler only on training features and transform
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler to transform test features
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Create k-NN graph (nodes = samples, edges = connections)
# Use only training data features here!
knn_graph = kneighbors_graph(X_train_scaled, n_neighbors=5, mode='connectivity', include_self=False)

# Get edge indices in COO format
edge_index_np = knn_graph.nonzero()
edge_index = np.array([edge_index_np[0], edge_index_np[1]])

# Function to convert to PyTorch Geometric Data object
def to_torch_data(X, y, edge_index):
    x = torch.tensor(X, dtype=torch.float)         # features tensor
    y = torch.tensor(y.values, dtype=torch.long)   # labels tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    return Data(x=x, y=y, edge_index=edge_index)

# Convert training data
train_data = to_torch_data(X_train_scaled, y_train, edge_index)

# For test data, graph construction usually differs because test nodes may not be connected same way.
# For simplicity, you can create a k-NN graph on test data as well or just use train edge_index.
# Here, we create a k-NN graph for test set similarly:

knn_graph_test = kneighbors_graph(X_test_scaled, n_neighbors=5, mode='connectivity', include_self=False)
edge_index_np_test = knn_graph_test.nonzero()
edge_index_test = np.array([edge_index_np_test[0], edge_index_np_test[1]])

test_data = to_torch_data(X_test_scaled, y_test, edge_index_test)


In [ ]:
class NonNegativeLinear(torch.nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = torch.nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x):
        weight_positive = F.relu(self.weight)  # Ensure non-negative weights
        return F.linear(x, weight_positive, self.bias)


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=0.75, gamma=2)

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops

class MonotonicGNNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(MonotonicGNNConv, self).__init__(aggr='add')  # sum aggregation
        self.lin = NonNegativeLinear(in_channels, out_channels)
        self.relu = torch.nn.ReLU()

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x = self.lin(x)  # apply non-negative linear transform
        return self.relu(self.propagate(edge_index, x=x))

    def message(self, x_j):
        return x_j

class MonotonicGNN(torch.nn.Module):
    def __init__(self, input_dim, output_dim=2, dropout=0.3):
        super(MonotonicGNN, self).__init__()
        # 6 layers instead of 2
        self.conv1 = MonotonicGNNConv(input_dim, 64)
        self.conv2 = MonotonicGNNConv(64, 128)
        self.conv3 = MonotonicGNNConv(128, 128)
        self.conv4 = MonotonicGNNConv(128, 64)
        self.conv5 = MonotonicGNNConv(64, 64)
        self.conv6 = MonotonicGNNConv(64, 32)

        self.dropout = torch.nn.Dropout(dropout)

        # Multi-task heads
        self.binary_head = torch.nn.Linear(32, 2)
        self.multiclass_head = torch.nn.Linear(32, 5)  # 5 attack types

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Skip connection 1
        x1 = self.dropout(self.conv1(x, edge_index))
        x2 = self.dropout(self.conv2(x1, edge_index))
        x2 = x2 + F.pad(x1, (0, 64))  # residual

        # Skip connection 2
        x3 = self.dropout(self.conv3(x2, edge_index))
        x4 = self.dropout(self.conv4(x3, edge_index))
        x4 = x4 + x2[:, :64]  # residual

        x5 = self.dropout(self.conv5(x4, edge_index))
        x6 = self.dropout(self.conv6(x5, edge_index))

        # Multi-task outputs
        binary_out = F.log_softmax(self.binary_head(x6), dim=1)
        multiclass_out = F.log_softmax(self.multiclass_head(x6), dim=1)

        return {'binary': binary_out, 'multiclass': multiclass_out}


In [ ]:
# Add multi-class labels
y_train_multi = df.iloc[X_train.index]['label'].apply(
    lambda x: {'normal':0,'DoS':1,'Probe':2,'R2L':3,'U2R':4}.get(x, 1)
)
y_test_multi = df.iloc[X_test.index]['label'].apply(
    lambda x: {'normal':0,'DoS':1,'Probe':2,'R2L':3,'U2R':4}.get(x, 1)
)

train_data.y_multi = torch.tensor(y_train_multi.values, dtype=torch.long)
test_data.y_multi = torch.tensor(y_test_multi.values, dtype=torch.long)

def train_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out = model(data)

    # Multi-task loss
    loss_binary = criterion(out['binary'], data.y)
    loss_multi = criterion(out['multiclass'], data.y_multi)
    loss = 0.6 * loss_binary + 0.4 * loss_multi

    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, data):
    model.eval()
    out = model(data)
    pred = out['binary'].argmax(dim=1)
    correct = (pred == data.y).sum().item()
    return correct / data.num_nodes

### MGNN training with explainability-

In [ ]:
import torch.optim as optim
model = MonotonicGNN(input_dim=X_train.shape[1], output_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=15, factor=0.5, min_lr=1e-6
)

best_loss = float('inf')
patience_counter = 0
patience = 25

for epoch in range(1, 501):
    loss = train_epoch(model, train_data, optimizer, criterion)
    test_acc = evaluate(model, test_data)

    scheduler.step(loss)

    # Early stopping
    if loss < best_loss:
        best_loss = loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d} | Loss: {loss:.4f} | Test Acc: {test_acc:.4f}')

# Load best model
model.load_state_dict(torch.load('best_model.pt'))

Epoch 010 | Loss: 284776071168.0000 | Test Acc: 0.4657
Epoch 020 | Loss: 112608288768.0000 | Test Acc: 0.5346
Epoch 030 | Loss: 57796255744.0000 | Test Acc: 0.4661
Epoch 040 | Loss: 31489908736.0000 | Test Acc: 0.4662
Epoch 050 | Loss: 19550330880.0000 | Test Acc: 0.5344
Epoch 060 | Loss: 13609062400.0000 | Test Acc: 0.4667
Epoch 070 | Loss: 9345376256.0000 | Test Acc: 0.4670
Epoch 080 | Loss: 6821928960.0000 | Test Acc: 0.5346
Epoch 090 | Loss: 5614390272.0000 | Test Acc: 0.4675
Epoch 100 | Loss: 4447580160.0000 | Test Acc: 0.4675
Epoch 110 | Loss: 3093445376.0000 | Test Acc: 0.4674
Epoch 120 | Loss: 2896467712.0000 | Test Acc: 0.5346
Epoch 130 | Loss: 2655452160.0000 | Test Acc: 0.4672
Epoch 140 | Loss: 2669694464.0000 | Test Acc: 0.5346
Epoch 150 | Loss: 1864931328.0000 | Test Acc: 0.5346
Epoch 160 | Loss: 2823148032.0000 | Test Acc: 0.5346
Early stopping at epoch 168


<All keys matched successfully>

In [ ]:
print(test_data.x.shape)

torch.Size([25195, 41])


In [ ]:
print("Conv1 weight importance per feature:\n", F.relu(model.conv1.lin.weight.data))

Conv1 weight importance per feature:
 tensor([[0.0000, 0.7603, 0.6609,  ..., 0.6510, 1.2643, 0.5167],
        [0.0527, 0.0000, 1.4223,  ..., 0.0000, 0.0153, 0.5106],
        [0.6237, 0.8920, 0.0000,  ..., 0.8565, 0.8453, 0.0000],
        ...,
        [0.0000, 0.4852, 0.0000,  ..., 0.3862, 0.0000, 1.3164],
        [0.1344, 0.1759, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.7664, 0.2219, 0.0000,  ..., 0.0000, 0.4973, 0.0000]])


In [ ]:
#Accuracy before tuning hyperparameters
from sklearn.metrics import precision_score, recall_score, f1_score
def evaluate_metrics(model, data):
    model.eval()
    out = model(data)
    preds = out.argmax(dim=1).cpu().numpy()
    labels = data.y.cpu().numpy()

    precision = precision_score(labels, preds, average='binary')
    recall = recall_score(labels, preds, average='binary')
    f1 = f1_score(labels, preds, average='binary')

    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return precision, recall, f1

# After training
evaluate_metrics(model, test_data)


Precision: 0.4662
Recall:    0.9987
F1 Score:  0.6356


(0.4661651142424966, 0.9987207914037183, 0.6356382978723404)

### Hyperparameter tuning

In [ ]:
import itertools

# Define hyperparameter grid
hidden_dims = [32,64,128]
learning_rates = [ 0.01,0.001]
dropouts = [0.0, 0.3, 0.5]

best_val_acc = 0
best_params = {}

for hidden_dim, lr, dropout in itertools.product(hidden_dims, learning_rates, dropouts):
    print(f"Training with hidden_dim={hidden_dim}, lr={lr}, dropout={dropout}")
    model = MonotonicGNN(input_dim=X_train.shape[1], hidden_dim=hidden_dim, output_dim=2, dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(200):
        loss = train_epoch(model, train_data, optimizer, criterion)

    val_acc = evaluate(model, test_data)  # use test_data or a separate validation set
    print(f"Validation Accuracy: {val_acc:.4f}\n")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = {'hidden_dim': hidden_dim, 'lr': lr, 'dropout': dropout}

print(f"Best Params: {best_params}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")


### Best Hyperparameters selected

In [ ]:
# Use the best hyperparameters you found
best_hidden_dim = 64
best_lr = 0.01
best_dropout = 0.3

# Instantiate the final model with best hyperparameters
model = MonotonicGNN(
    input_dim=X_train.shape[1],
    hidden_dim=best_hidden_dim,
    output_dim=2,
    dropout=best_dropout
)

optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
criterion = torch.nn.CrossEntropyLoss()

epochs = 300 # or whatever you want for full training

for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_data, optimizer, criterion)

    if epoch % 10 == 0:
        train_acc = evaluate(model, train_data)
        test_acc = evaluate(model, test_data)
        print(f'Epoch {epoch:03d} | Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}')




In [ ]:
#Accuracy after tuning hyperparameters
from sklearn.metrics import precision_score, recall_score, f1_score
def evaluate_metrics(model, data):
    model.eval()
    out = model(data)
    preds = out.argmax(dim=1).cpu().numpy()
    labels = data.y.cpu().numpy()

    precision = precision_score(labels, preds, average='binary')
    recall = recall_score(labels, preds, average='binary')
    f1 = f1_score(labels, preds, average='binary')

    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return precision, recall, f1

# After training
evaluate_metrics(model, test_data)


Precision: 0.4662
Recall:    0.9987
F1 Score:  0.6356


(0.4661651142424966, 0.9987207914037183, 0.6356382978723404)

In [ ]:
print("Conv1 weight importance per feature:\n", F.relu(model.conv1.lin.weight.data))

Conv1 weight importance per feature:
 tensor([[0.0000, 0.7603, 0.6609,  ..., 0.6510, 1.2643, 0.5167],
        [0.0527, 0.0000, 1.4223,  ..., 0.0000, 0.0153, 0.5106],
        [0.6237, 0.8920, 0.0000,  ..., 0.8565, 0.8453, 0.0000],
        ...,
        [0.0000, 0.4852, 0.0000,  ..., 0.3862, 0.0000, 1.3164],
        [0.1344, 0.1759, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.7664, 0.2219, 0.0000,  ..., 0.0000, 0.4973, 0.0000]])
